<a href="https://www.kaggle.com/code/avikdas567/predicting-heart-disease?scriptVersionId=296036037" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# =========================================================
# Kaggle Playground S6E2 - Predicting Heart Disease
# =========================================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier
import lightgbm as lgb

# =====================
# Load Data
# =====================
train = pd.read_csv("/kaggle/input/playground-series-s6e2/train.csv")
test  = pd.read_csv("/kaggle/input/playground-series-s6e2/test.csv")

TARGET = "Heart Disease"
ID_COL = "id"

X = train.drop(columns=[TARGET])

y = train[TARGET].map({
    "Absence": 0,
    "Presence": 1
}).astype(int)

X_test = test.copy()

# =====================
# Encode Categorical Features
# =====================
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    full = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(full)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

# =====================
# Cross Validation
# =====================
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

# =====================
# Models
# =====================
cat_params = {
    "iterations": 2500,
    "learning_rate": 0.03,
    "depth": 8,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 0,
    "early_stopping_rounds": 200
}

lgb_params = {
    "objective": "binary",
    "metric": "auc",
    "learning_rate": 0.03,
    "num_leaves": 64,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "seed": 42,
    "verbosity": -1
}

print("Training started...\n")

# =====================
# Training Loop
# =====================
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"Fold {fold}")

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # ---- CatBoost ----
    cat_model = CatBoostClassifier(**cat_params)
    cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

    val_cat = cat_model.predict_proba(X_val)[:, 1]
    test_cat = cat_model.predict_proba(X_test)[:, 1]

    # ---- LightGBM ----
    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_valid = lgb.Dataset(X_val, y_val)

    lgb_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_valid],
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
    )

    val_lgb = lgb_model.predict(X_val)
    test_lgb = lgb_model.predict(X_test)

    # ---- Ensemble ----
    val_pred = 0.5 * val_cat + 0.5 * val_lgb
    test_pred = 0.5 * test_cat + 0.5 * test_lgb

    oof_preds[val_idx] = val_pred
    test_preds += test_pred / N_SPLITS

    fold_auc = roc_auc_score(y_val, val_pred)
    print(f"  AUC: {fold_auc:.5f}")

# =====================
# Final CV
# =====================
cv_auc = roc_auc_score(y, oof_preds)
print("\n====================")
print(f"CV ROC-AUC: {cv_auc:.6f}")
print("====================\n")

# =====================
# Submission
# =====================
submission = pd.DataFrame({
    "id": test[ID_COL],
    "Heart Disease": test_preds
})

submission.to_csv("submission.csv", index=False)
print("submission.csv saved successfully!")
display(submission.head())

Training started...

Fold 1
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[631]	valid_0's auc: 0.955505
  AUC: 0.95566
Fold 2
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[600]	valid_0's auc: 0.954598
  AUC: 0.95469
Fold 3
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[692]	valid_0's auc: 0.955288
  AUC: 0.95544
Fold 4
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[744]	valid_0's auc: 0.954805
  AUC: 0.95496
Fold 5
Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[655]	valid_0's auc: 0.955699
  AUC: 0.95581

CV ROC-AUC: 0.955307

submission.csv saved successfully!


,id,Heart Disease
0,630000,0.940228
1,630001,0.008427
2,630002,0.986196
3,630003,0.004489
4,630004,0.196709
